In [1]:
text = "This is some text"
byte_ary = bytearray(text, "utf-8")
print(byte_ary)

bytearray(b'This is some text')


In [2]:
ids = list(byte_ary)
print(ids)

[84, 104, 105, 115, 32, 105, 115, 32, 115, 111, 109, 101, 32, 116, 101, 120, 116]


- Text can be converted to bytearray
- casting to a list give the list of bytes that is integer

In [3]:
print("Number of characters:", len(text))
print("Number of token IDs:", len(ids))

Number of characters: 17
Number of token IDs: 17


- Number of id could be different because some chars need more bytes to represent them
- Bytes value range is 0-255 ($2^8$)

In [5]:
text = "marché"
print(list(bytearray(text, encoding="utf-8")))

[109, 97, 114, 99, 104, 195, 169]


The algorithm
- Find most frequent adjacent pair of bytes (a, b)
- Replace the pair (a, b) with new ID
- Repeat until no gain
- Decoding: reverse the process by substituating the id by the correspond pair

In [6]:
text = "the cat in the hat"
text_bytes = list(bytearray(text, encoding="utf-8"))
text_bytes

[116,
 104,
 101,
 32,
 99,
 97,
 116,
 32,
 105,
 110,
 32,
 116,
 104,
 101,
 32,
 104,
 97,
 116]

- text: `the cat in the hat`

In [10]:
from collections import Counter
from typing import Sequence
def get_pair_count(ids: Sequence[int]):
    pairs = zip(ids, ids[1:])
    return Counter(pairs)

In [11]:

counts = get_pair_count(text_bytes)

In [14]:
counts

Counter({(116, 104): 2,
         (104, 101): 2,
         (101, 32): 2,
         (97, 116): 2,
         (32, 99): 1,
         (99, 97): 1,
         (116, 32): 1,
         (32, 105): 1,
         (105, 110): 1,
         (110, 32): 1,
         (32, 116): 1,
         (32, 104): 1,
         (104, 97): 1})

- 0-255 is reserved for all possible single bytes
- Each iteration, new_id = 256 + i, i=0, i=total_iter

In [13]:
top_pair = counts.most_common(1)[0]
top_pair

((116, 104), 2)

- Iterate overall
- The pair is replaced by the new id and other is kept as is
- 

In [16]:
def merge(ids: Sequence[int], pair: tuple[int, int], new_id: int):
    new_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            new_ids.append(new_id)
            i += 2          # skip both elements of the pair
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids


The loop

In [20]:
vocab_size = 276 # 256 + 20 merge
ids = text_bytes
merges = {}
for i in range(vocab_size-256):
    counts = get_pair_count(ids=ids)
    if not counts:
        break
    top_pair = max(counts, key=counts.get)
    new_id = 256 + i
    ids = merge(ids=ids, pair=top_pair, new_id=new_id)
    merges[top_pair] = new_id


- Need to track merge str and concat them
- To decode, we take a list of int in the vocab and return the corresponding string
- After the train, how to encode new text?

## Reference

- https://sebastianraschka.com/blog/2025/bpe-from-scratch.html